# 03 - Model Training
ConvLSTM2D spatiotemporal model: 3-month sequences -> 1-month fishing ground prediction.

**Input (from Notebook 02):** `preprocessed_features.nc` (7 channels, 41x25 grid, monthly)

**Outputs:** `convlstm_model.keras`, `best_model.keras`, `X_test.npy`, `y_test.npy`, `training_history.json`, `data_summary.json`

In [1]:
!pip install -q tensorflow

In [2]:
import xarray as xr
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import ConvLSTM2D, Conv2D, BatchNormalization, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import json, os
# CELL: imports, drive mount, and shared pipeline CONFIG
from google.colab import drive

drive.mount('/content/drive')

CONFIG = {
    # ------------------------------------------------------------------
    # Spatial
    # ------------------------------------------------------------------
    # Broad staging bbox (Notebook 01 raw cache)
    'bbox_regional': {
        'lat_min': 0,   'lat_max': 30,
        'lon_min': 110, 'lon_max': 140,
    },
    # WPS model bbox (Notebook 02 onward)
    'bbox_model': {
        'lat_min': 10,  'lat_max': 20,
        'lon_min': 114, 'lon_max': 120,
    },

    # ------------------------------------------------------------------
    # Temporal
    # ------------------------------------------------------------------
    'date_full':  {'start': '2014-01-01', 'end': '2024-12-31'},
    'date_model': {'start': '2019-01-01', 'end': '2024-12-31'},

    # ------------------------------------------------------------------
    # Paths
    # ------------------------------------------------------------------
    'data_dir': '/content/drive/MyDrive/fishing_project/',
    'files': {
        # Source CMEMS NetCDF files placed in data_dir by the user
        'physics_w_nc':  'cmems_mod_glo_phy_my_0.083deg_P1M-m_1779636039565.nc',
        'physics_ht_nc': 'cmems_mod_glo_phy_my_0.083deg_P1M-m_1779635380319.nc',
        'bgc_src_nc':    'cmems_mod_glo_bgc_my_0.25deg_P1M-m_1779635372583.nc',
        # 01 outputs -> 02 inputs
        'physics_nc':    'physics_raw_region.nc',
        'bgc_nc':        'bgc_raw_region.nc',
        'ais_parquet':   'ais_raw_region.parquet',
        'ais_csv_gz':    'ais_raw_region.csv.gz',
        # 02 outputs -> 03 inputs
        'ais_gridded_nc':  'ais_fishing_effort_gridded.nc',
        'preprocessed_nc': 'preprocessed_features.nc',
        # 03 outputs -> 04 / 05 / dashboard inputs
        'model_keras':     'convlstm_model.keras',
        'best_model':      'best_model.keras',
        'X_test_npy':      'X_test.npy',
        'y_test_npy':      'y_test.npy',
        'history_json':    'training_history.json',
        'summary_json':    'data_summary.json',
        # 04 outputs -> 05 / dashboard inputs
        'predictions_npy': 'predictions.npy',
        'eval_csv':        'evaluation_results.csv',
    },

    # ------------------------------------------------------------------
    # AIS
    # ------------------------------------------------------------------
    'ais_use_cols': ['date', 'cell_ll_lat', 'cell_ll_lon', 'fishing_hours'],

    # ------------------------------------------------------------------
    # Depth selection for CMEMS variables
    # ------------------------------------------------------------------
    'physics_surface_depth': 0.49,   # metres, nearest-neighbour selection
    'bgc_depth_range': (0.51, 5.14), # metres, averaged over this range

    # ------------------------------------------------------------------
    # Preprocessing
    # ------------------------------------------------------------------
    'norm_method':   'minmax',
    'resample_freq': '1ME',

    # ------------------------------------------------------------------
    # Model / Training
    # ------------------------------------------------------------------
    'seq_len':    3,     # months of input context fed to ConvLSTM
    'pred_len':   1,     # months ahead to predict
    'n_channels': 7,     # sst, ssh, vo, uo, chl, nppv, fishing_effort
    'train_frac': 0.70,
    'val_frac':   0.15,
    # test_frac = 1 - 0.70 - 0.15 = 0.15
    'epochs':     50,
    'batch_size': 8,
    'patience':   10,

    # ------------------------------------------------------------------
    # Evaluation
    # ------------------------------------------------------------------
    'f1_threshold': 0.15,   # binarization threshold for F1 score in 04_evaluation
}

DATA_DIR   = CONFIG['data_dir']
f          = CONFIG['files']


Mounted at /content/drive


In [3]:
# CELL: load preprocessed features from Notebook 02
SEQ_LEN    = CONFIG['seq_len']
PRED_LEN   = CONFIG['pred_len']
N_CHANNELS = CONFIG['n_channels']

features = xr.open_dataset(DATA_DIR + f['preprocessed_nc'])
n_months = features.sizes['time']
n_lat    = features.sizes['latitude']
n_lon    = features.sizes['longitude']

print(f'Preprocessed features:')
print(f'  time={n_months}, latitude={n_lat}, longitude={n_lon}')
print(f'  Variables: {list(features.data_vars)}')
# Expected: ['chl', 'nppv', 'ssh', 'sst', 'uo', 'vo', 'fishing_effort']


Preprocessed features:
  time=72, latitude=41, longitude=25
  Variables: ['chl', 'nppv', 'ssh', 'sst', 'uo', 'vo', 'fishing_effort']


In [4]:
# CELL: stack all 7 channels into a single NumPy array
# Channel order matches Full_ConvLSTM.py CHANNELS list: sst,ssh,vo,uo,chl,nppv,fishing_effort
X_stack = np.stack([
    features['sst'].values,            # Ch 0 - Sea surface temperature
    features['ssh'].values,            # Ch 1 - Sea surface height
    features['vo'].values,             # Ch 2 - Northward sea water velocity
    features['uo'].values,             # Ch 3 - Eastward sea water velocity
    features['chl'].values,            # Ch 4 - Chlorophyll-a
    features['nppv'].values,           # Ch 5 - Net primary production
    features['fishing_effort'].values, # Ch 6 - AIS fishing effort (log-norm)
], axis=-1)  # (n_months, n_lat, n_lon, 7)

y_array = features['fishing_effort'].values  # (n_months, n_lat, n_lon)

# Replace NaNs (land/masked cells) with 0 before sequence generation
X_stack = np.nan_to_num(X_stack, nan=0.0)
y_array = np.nan_to_num(y_array, nan=0.0)

print(f'X stack shape : {X_stack.shape}')   # Expected: (72, 41, 25, 7)
print(f'y array shape : {y_array.shape}')   # Expected: (72, 41, 25)


X stack shape : (72, 41, 25, 7)
y array shape : (72, 41, 25)


In [5]:
# CELL: create sliding-window sequences (seq_len months in -> pred_len months ahead)
def create_sequences(X, y, seq_len, pred_len):
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_len - pred_len + 1):
        X_seq.append(X[i : i + seq_len])
        y_seq.append(y[i + seq_len : i + seq_len + pred_len])
    return np.array(X_seq), np.array(y_seq)

X_seq, y_seq = create_sequences(X_stack, y_array, SEQ_LEN, PRED_LEN)
print(f'Sequences : {X_seq.shape} -> {y_seq.shape}')
# Expected: (69, 3, 41, 25, 7) -> (69, 1, 41, 25)


Sequences : (69, 3, 41, 25, 7) -> (69, 1, 41, 25)


In [6]:
# CELL: chronological train / val / test split (70 / 15 / 15)
train_size = int(CONFIG['train_frac'] * len(X_seq))
val_size   = int(CONFIG['val_frac']   * len(X_seq))

X_train = X_seq[:train_size]
y_train = y_seq[:train_size]
X_val   = X_seq[train_size : train_size + val_size]
y_val   = y_seq[train_size : train_size + val_size]
X_test  = X_seq[train_size + val_size :]
y_test  = y_seq[train_size + val_size :]

print(f'Train : {X_train.shape}')
print(f'Val   : {X_val.shape}')
print(f'Test  : {X_test.shape}')

# Save X_test immediately (unsqueezed, for 04_evaluation compatibility)
np.save(DATA_DIR + f['X_test_npy'], X_test)
print(f'Saved X_test -> {f["X_test_npy"]}')


Train : (48, 3, 41, 25, 7)
Val   : (10, 3, 41, 25, 7)
Test  : (11, 3, 41, 25, 7)
Saved X_test -> X_test.npy


In [7]:
# CELL: reshape y arrays to match ConvLSTM output shape (N, lat, lon, 1)
y_train      = y_train.squeeze(axis=1)[..., np.newaxis]   # (N_train, 41, 25, 1)
y_val        = y_val.squeeze(axis=1)[..., np.newaxis]     # (N_val,   41, 25, 1)
y_test_model = y_test.squeeze(axis=1)[..., np.newaxis]    # (N_test,  41, 25, 1)

# Overwrite y_test with final (N, 41, 25, 1) shape --
# 04_evaluation and the dashboard both load this shape.
np.save(DATA_DIR + f['y_test_npy'], y_test_model)
print(f'Saved y_test -> {f["y_test_npy"]}  shape={y_test_model.shape}')

print(f'y_train : {y_train.shape}')
print(f'y_val   : {y_val.shape}')


Saved y_test -> y_test.npy  shape=(11, 41, 25, 1)
y_train : (48, 41, 25, 1)
y_val   : (10, 41, 25, 1)


In [8]:
# CELL: build ConvLSTM2D model
# Uses keras.Input() as first layer to suppress legacy UserWarning about input_shape.
model = Sequential([
    Input(shape=(SEQ_LEN, n_lat, n_lon, N_CHANNELS)),
    ConvLSTM2D(
        filters=64, kernel_size=(3, 3),
        padding='same', return_sequences=True,
    ),
    BatchNormalization(),
    Dropout(0.2),
    ConvLSTM2D(
        filters=32, kernel_size=(3, 3),
        padding='same', return_sequences=False,
    ),
    BatchNormalization(),
    Dropout(0.2),
    Conv2D(filters=1, kernel_size=(1, 1), activation='sigmoid', padding='same'),
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['mae'],
)
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv_lstm2d (ConvLSTM2D)        │ (None, 3, 41, 25, 64)  │       163,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 3, 41, 25, 64)  │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 3, 41, 25, 64)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_lstm2d_1 (ConvLSTM2D)      │ (None, 41, 25, 32)     │       110,720 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 41, 25, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 41, 25, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 41, 25, 1)      │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 274,977 (1.05 MB)

 Trainable params: 274,785 (1.05 MB)

 Non-trainable params: 192 (768.00 B)

In [9]:
# CELL: define callbacks
# ModelCheckpoint saves to .keras (native Keras format, avoids legacy HDF5 warning).
best_model_path  = DATA_DIR + f['best_model']
final_model_path = DATA_DIR + f['model_keras']

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=CONFIG['patience'],
        restore_best_weights=True,
    ),
    ModelCheckpoint(
        best_model_path,
        monitor='val_loss',
        save_best_only=True,
    ),
]
print(f'Best model checkpoint : {best_model_path}')
print(f'Final model path      : {final_model_path}')


Best model checkpoint : /content/drive/MyDrive/fishing_project/best_model.keras
Final model path      : /content/drive/MyDrive/fishing_project/convlstm_model.keras


In [10]:
# CELL: train model
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=CONFIG['epochs'],
    batch_size=CONFIG['batch_size'],
    callbacks=callbacks,
    verbose=1,
)

best_epoch    = int(np.argmin(history.history['val_loss'])) + 1
best_val_loss = float(np.min(history.history['val_loss']))
n_epochs_run  = len(history.history['loss'])
print(f'Training complete: {n_epochs_run} epochs, '
      f'best val_loss={best_val_loss:.6f} at epoch {best_epoch}')


Epoch 1/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 16s 1s/step - loss: 0.8566 - mae: 0.4741 - val_loss: 0.6671 - val_mae: 0.3870
Epoch 2/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - loss: 0.7357 - mae: 0.4667 - val_loss: 0.6595 - val_mae: 0.3827
Epoch 3/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - loss: 0.6991 - mae: 0.4532 - val_loss: 0.6440 - val_mae: 0.3739
Epoch 4/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - loss: 0.6729 - mae: 0.4408 - val_loss: 0.6321 - val_mae: 0.3671
Epoch 5/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - loss: 0.6519 - mae: 0.4336 - val_loss: 0.6205 - val_mae: 0.3603
Epoch 6/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - loss: 0.6299 - mae: 0.4231 - val_loss: 0.6036 - val_mae: 0.3504
Epoch 7/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - loss: 0.6100 - mae: 0.4128 - val_loss: 0.5939 - val_mae: 0.3447
Epoch 8/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - loss: 0.5922 - mae: 0.4038 - val_loss: 0.5834 - val_mae: 0.3384
Epoch 9/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - loss: 0.5733 - mae: 0.3944 - v

In [11]:
# CELL: save model + history + data summary
# ------------------------------------------------------------------
# 1. Final model (native .keras, no HDF5 warnings)
# ------------------------------------------------------------------
model.save(final_model_path)
print(f'Saved model   : {final_model_path}')

# ------------------------------------------------------------------
# 2. training_history.json -- read by Full_ConvLSTM.py dashboard
# ------------------------------------------------------------------
history_dict = {k: [float(v) for v in vals]
                for k, vals in history.history.items()}
with open(DATA_DIR + f['history_json'], 'w') as fh:
    json.dump(history_dict, fh, indent=2)
print(f'Saved history : {f["history_json"]}')

# ------------------------------------------------------------------
# 3. data_summary.json -- pipeline metadata for dashboard and reproducibility
# ------------------------------------------------------------------
bm = CONFIG['bbox_model']
dm = CONFIG['date_model']
summary = {
    'date_range':    f"{dm['start']} to {dm['end']}",
    'n_months':      int(n_months),
    'n_lat':         int(n_lat),
    'n_lon':         int(n_lon),
    'n_channels':    int(N_CHANNELS),
    'seq_len':       int(SEQ_LEN),
    'pred_len':      int(PRED_LEN),
    'bbox_model':    bm,
    'n_train':       int(X_train.shape[0]),
    'n_val':         int(X_val.shape[0]),
    'n_test':        int(X_test.shape[0]),
    'best_epoch':    best_epoch,
    'best_val_loss': best_val_loss,
}
with open(DATA_DIR + f['summary_json'], 'w') as fh:
    json.dump(summary, fh, indent=2)
print(f'Saved summary : {f["summary_json"]}')
print('Notebook 03 complete. Run 04_evaluation.ipynb next.')


Saved model   : /content/drive/MyDrive/fishing_project/convlstm_model.keras
Saved history : training_history.json
Saved summary : data_summary.json
Notebook 03 complete. Run 04_evaluation.ipynb next.
